In [ ]:
# =======================
# Notebook 一体化：XGBoost 逐列机器学习插补（兼容旧版 xgboost，无 early_stopping_rounds）
# =======================
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score

# xgboost 基本导入
try:
    import xgboost as xgb
    from xgboost import XGBClassifier, XGBRegressor
    import lightgbm as lgb
    from lightgbm import LGBMClassifier, LGBMRegressor
except Exception as e:
    raise RuntimeError("需要已安装 xgboost。请先在该环境安装：pip install xgboost") from e

# ============ Config（按需修改） ============
INPUT_CSV  = "/content/drive/MyDrive/demo_200000.csv"
OUTPUT_CSV = "demo_200000_imputed.csv"
REPORT_CSV = "demo_200000_impute_report.csv"

# 不参与作为特征/目标的列（如ID/标签）
EXCLUDE_COLS = ["ID", "DR","DR_time","AMD_time","AMD","glaucoma_time","glaucoma","cataract_time","cataract"]

# 类别压帽（最多保留前N个高频类别，其余合并为 OTHER）
MAX_CATEGORIES = 50

# 训练与评估
TEST_SIZE = 0.2
RANDOM_STATE = 42
N_ESTIMATORS = 300           # 如果耗时长，可先降到 200
LEARNING_RATE = 0.05
MAX_DEPTH = 6
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 0.8

# 早停轮数（老版本不支持 fit 参数，我们用回调；若回调也不可用就自动跳过）
EARLY_STOPPING_ROUNDS = 50

# 分类任务：最低可接受准确率（低于则回退到众数）
EVAL_ACC_THRESHOLD = 0.65
# 回归任务：模型MSE需 <= baseline*MSE_RATIO 才接受（即至少优于均值/中位数约5%）
EVAL_MSE_RATIO = 0.95

# XGBoost tree_method（可切到 "gpu_hist" 如果你的环境支持 GPU）
XGB_TREE_METHOD = "hist"     # "hist" | "approx" | "auto" | "gpu_hist"


# ============ Helpers ============
def is_numeric_series(s: pd.Series) -> bool:
    return pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s)

def cap_categories(series: pd.Series, max_categories: int = 50):
    """保留前N高频类别，其余 -> 'OTHER'"""
    vc = series.value_counts(dropna=False)
    top = set(vc.head(max_categories).index.tolist())
    return series.apply(lambda x: x if x in top else "OTHER")

def one_hot_fit_transform(df: pd.DataFrame, categorical_cols, max_categories: int):
    """拟合并独热编码，返回：编码后DF、meta（每列类别集合）、最终列名列表"""
    df = df.copy()
    meta = {}
    for col in categorical_cols:
        s = df[col].astype(str).fillna("UNKNOWN")
        s = cap_categories(s, max_categories=max_categories)
        df[col] = s
        meta[col] = sorted(df[col].unique().tolist())
    dummied = pd.get_dummies(df, columns=categorical_cols, dummy_na=False)
    return dummied, meta, dummied.columns.tolist()

def one_hot_transform_with_meta(df: pd.DataFrame, categorical_cols, meta, all_cols):
    """用拟合阶段的 meta 做独热，并对齐列"""
    df = df.copy()
    for col in categorical_cols:
        s = df[col].astype(str).fillna("UNKNOWN")
        df[col] = s.apply(lambda x: x if x in meta[col] else "OTHER")
    dummied = pd.get_dummies(df, columns=categorical_cols, dummy_na=False)
    for c in all_cols:
        if c not in dummied.columns:
            dummied[c] = 0
    dummied = dummied[all_cols]
    return dummied

def evaluate_classifier(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    return {"accuracy": acc, "f1_macro": f1m}

def evaluate_regressor(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    return {"mse": mse, "r2": r2}

def _fit_with_optional_early_stopping(model, X_tr, y_tr, X_va, y_va):
    """
    兼容不同 xgboost 版本的早停：
    - 优先使用 xgboost.callback.EarlyStopping
    - 如果不可用，直接不做早停
    """
    callbacks = []
    eval_set = [(X_va, y_va)]

    # 优先用官方回调（老版本也支持）
    try:
        cb = xgb.callback.EarlyStopping(
            rounds=EARLY_STOPPING_ROUNDS,
            save_best=True,
            maximize=False  # 回归/分类默认都是最小化损失
        )
        callbacks.append(cb)
        model.fit(X_tr, y_tr, eval_set=eval_set, callbacks=callbacks, verbose=False)
        return model
    except Exception:
        # 无法使用回调则直接无早停训练
        model.fit(X_tr, y_tr, eval_set=eval_set, verbose=False)
        return model

def xgb_impute_column(
    df: pd.DataFrame,
    target_col: str,
    exclude_cols: list,
    max_categories: int = 50,
    test_size: float = 0.2,
    random_state: int = 42,
    n_estimators: int = 300,
    learning_rate: float = 0.05,
    max_depth: int = 6,
    subsample: float = 0.8,
    colsample_bytree: float = 0.8,
    eval_acc_threshold: float = 0.65,
    eval_mse_ratio: float = 0.95,
    tree_method: str = "hist",
):
    """对单列进行 XGB 插补：返回填补后的 Series 与一条报告 dict"""
    y = df[target_col]
    notnull_mask = y.notna()
    null_mask = ~notnull_mask
    if null_mask.sum() == 0:
        return y, {"column": target_col, "type": "skip_no_missing", "trained": False,
                   "metric_primary": None, "metric_secondary": None, "fallback": "none"}

    feature_cols = [c for c in df.columns if c != target_col and c not in exclude_cols]
    X = df[feature_cols].copy()

    # 特征侧预填（不改原 df）
    num_cols = [c for c in feature_cols if is_numeric_series(X[c])]
    cat_cols = [c for c in feature_cols if not is_numeric_series(X[c])]
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype(str).fillna("UNKNOWN")

    X_train_full = X.loc[notnull_mask].copy()
    y_train_full = y.loc[notnull_mask].copy()
    X_null       = X.loc[null_mask].copy()

    X_train_oh, meta, all_cols = one_hot_fit_transform(X_train_full, cat_cols, max_categories=max_categories)
    X_null_oh = one_hot_transform_with_meta(X_null, cat_cols, meta, all_cols)

    # 回归还是分类由目标列类型决定
    if is_numeric_series(y_train_full):
        task = "regression"
        y_vec = pd.to_numeric(y_train_full, errors="coerce")
        valid = y_vec.notna()
        X_train_oh = X_train_oh.loc[valid]; y_vec = y_vec.loc[valid]

        X_tr, X_te, y_tr, y_te = train_test_split(X_train_oh, y_vec, test_size=test_size, random_state=random_state)
        model = XGBRegressor(
            n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth,
            subsample=subsample, colsample_bytree=colsample_bytree, objective="reg:squarederror",
            tree_method=tree_method, random_state=random_state, n_jobs=-1
        )
        model = _fit_with_optional_early_stopping(model, X_tr, y_tr, X_te, y_te)
        y_pred = model.predict(X_te)
        m = evaluate_regressor(y_te, y_pred)

        # 基线：均值/中位数
        mse_model = m["mse"]
        mse_mean  = mean_squared_error(y_te, np.full_like(y_te, y_tr.mean(), dtype=float))
        mse_median= mean_squared_error(y_te, np.full_like(y_te, float(np.median(y_tr)), dtype=float))
        best_bl = min(mse_mean, mse_median)

        if mse_model <= eval_mse_ratio * best_bl:
            y_null_pred = model.predict(X_null_oh)
            filled = y.copy(); filled.loc[null_mask] = y_null_pred
            rep = {"column": target_col, "type": task, "trained": True,
                   "metric_primary": f"mse={m['mse']:.5f}", "metric_secondary": f"r2={m['r2']:.4f}",
                   "fallback": "none"}
            return filled, rep
        else:
            filled = y.fillna(y.median())
            rep = {"column": target_col, "type": task, "trained": False,
                   "metric_primary": f"mse={m['mse']:.5f}", "metric_secondary": f"r2={m['r2']:.4f}",
                   "fallback": "median"}
            return filled, rep

    else:
        task = "classification"
        y_str = y_train_full.astype(str)
        enc = LabelEncoder(); y_enc = enc.fit_transform(y_str)

        X_tr, X_te, y_tr, y_te = train_test_split(
            X_train_oh, y_enc, test_size=test_size, random_state=random_state, stratify=y_enc
        )
        clf = XGBClassifier(
            n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth,
            subsample=subsample, colsample_bytree=colsample_bytree,
            objective="multi:softprob" if len(np.unique(y_enc))>2 else "binary:logistic",
            num_class=len(np.unique(y_enc)) if len(np.unique(y_enc))>2 else None,
            tree_method=tree_method, random_state=random_state, n_jobs=-1, use_label_encoder=False
        )
        clf = _fit_with_optional_early_stopping(clf, X_tr, y_tr, X_te, y_te)
        y_pred = clf.predict(X_te)
        m = evaluate_classifier(y_te, y_pred)

        if m["accuracy"] >= eval_acc_threshold:
            y_null_pred_enc = clf.predict(X_null_oh)
            y_null_pred = enc.inverse_transform(y_null_pred_enc)
            filled = y.copy(); filled.loc[null_mask] = y_null_pred
            rep = {"column": target_col, "type": task, "trained": True,
                   "metric_primary": f"acc={m['accuracy']:.4f}", "metric_secondary": f"f1_macro={m['f1_macro']:.4f}",
                   "fallback": "none"}
            return filled, rep
        else:
            mode_val = y_str.mode().iloc[0] if y_str.mode().shape[0] else "UNKNOWN"
            filled = y.fillna(mode_val)
            rep = {"column": target_col, "type": task, "trained": False,
                   "metric_primary": f"acc={m['accuracy']:.4f}", "metric_secondary": f"f1_macro={m['f1_macro']:.4f}",
                   "fallback": f"mode({mode_val})"}
            return filled, rep


def impute_dataframe(df: pd.DataFrame, exclude_cols=None, max_categories=50):
    dfw = df.copy()
    exclude_cols = exclude_cols or []

    miss_counts = dfw.isna().sum()
    miss_cols = miss_counts[miss_counts > 0].sort_values(ascending=True).index.tolist()

    reports = []
    for col in miss_cols:
        if col in exclude_cols:
            reports.append({"column": col, "type": "skipped_excluded", "trained": False,
                            "metric_primary": None, "metric_secondary": None, "fallback": "none"})
            continue

        filled_col, rep = xgb_impute_column(
            df=dfw, target_col=col, exclude_cols=exclude_cols,
            max_categories=max_categories, test_size=TEST_SIZE, random_state=RANDOM_STATE,
            n_estimators=N_ESTIMATORS, learning_rate=LEARNING_RATE, max_depth=MAX_DEPTH,
            subsample=SUBSAMPLE, colsample_bytree=COLSAMPLE_BYTREE,
            eval_acc_threshold=EVAL_ACC_THRESHOLD, eval_mse_ratio=EVAL_MSE_RATIO,
            tree_method=XGB_TREE_METHOD
        )
        dfw[col] = filled_col
        reports.append(rep)
        print(f"[OK] {col}: {rep}")

    report_df = pd.DataFrame(reports, columns=["column","type","trained","metric_primary","metric_secondary","fallback"])
    return dfw, report_df


# ============ RUN ============
df_in = pd.read_csv(INPUT_CSV)
imputed_df, report_df = impute_dataframe(df_in, exclude_cols=EXCLUDE_COLS, max_categories=MAX_CATEGORIES)

imputed_df.to_csv(OUTPUT_CSV, index=False)
report_df.to_csv(REPORT_CSV, index=False)

print("插补完成。")
print("Imputed CSV ->", OUTPUT_CSV)
print("Report CSV  ->", REPORT_CSV)

# 预览
imputed_df.head(), report_df.head(20)


[OK] Ownsend_Deprivation_Index: {'column': 'Ownsend_Deprivation_Index', 'type': 'regression', 'trained': True, 'metric_primary': 'mse=3.03671', 'metric_secondary': 'r2=0.6825', 'fallback': 'none'}
[OK] Number_of_Self-Reported_Cancers: {'column': 'Number_of_Self-Reported_Cancers', 'type': 'regression', 'trained': True, 'metric_primary': 'mse=0.01536', 'metric_secondary': 'r2=0.8008', 'fallback': 'none'}
[OK] Operations: {'column': 'Operations', 'type': 'regression', 'trained': True, 'metric_primary': 'mse=1.65087', 'metric_secondary': 'r2=0.3075', 'fallback': 'none'}
[OK] Number_of_Treatments/Medications: {'column': 'Number_of_Treatments/Medications', 'type': 'regression', 'trained': True, 'metric_primary': 'mse=2.40922', 'metric_secondary': 'r2=0.6692', 'fallback': 'none'}
[OK] Number_of_Self-Reported_Non-Cancer_Illnesses: {'column': 'Number_of_Self-Reported_Non-Cancer_Illnesses', 'type': 'regression', 'trained': True, 'metric_primary': 'mse=1.53121', 'metric_secondary': 'r2=0.5638', '